## models

In [1]:
import quante as qt

# xxx
qt.generate.matrix.heisenberg_matrix(
    L=10, j=(1.,1.,1.), h=0., pauli=True
)

array([[9., 0., 0., ..., 0., 0., 0.],
       [0., 7., 2., ..., 0., 0., 0.],
       [0., 2., 5., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 5., 2., 0.],
       [0., 0., 0., ..., 2., 7., 0.],
       [0., 0., 0., ..., 0., 0., 9.]])

In [2]:
# Dirac Fermion Model

import quante as qt

qt.generate.matrix.syk4_dirac(
    L=10, Nf=5, J=1.
)

array([[-0.15998976+0.j        ,  0.04726786-0.02488769j,  0.06282416+0.02705359j, ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [ 0.04726786+0.02488769j, -0.39299243+0.j        ,  0.08404552-0.07282302j, ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [ 0.06282416-0.02705359j,  0.08404552+0.07282302j, -0.35937867+0.j        , ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       ...,
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ..., -0.21594568+0.j        ,  0.04341572+0.08007936j, -0.02433892-0.01025381j],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ...,  0.04341572-0.08007936j,  0.01818395+0.j        , -0.09943106-0.01004965j],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ..., -0.02433892+0.01025381j, -0.09943106+0.01004965j, -0.00683021+0.j        ]])

## 基本使用

定义哈密顿量：

$$
    H = \sum_{i = 1}^{L - 1} (\sigma^{x}_{i}\sigma^{x}_{i + 1} + \sigma^{y}_{i} \sigma^{y}_{i + 1} + \frac{1}{2}  \sigma^{z}_{i} \sigma^{z}_{i + 1})
$$

In [3]:
import quante as qt
op = qt.generate.operas.spin
L = 4
builder = op.builder()
for i in range(L-1):
    builder += 'xx', [i, i+1], 1.
    builder += 'yy', [i, i+1], 1.
    builder += 'zz', [i, i+1], 1.
ham = builder.build()

basis = qt.generate.basis.spin_basis(L=L, Nup=L//2)
hammat = ham.to_matrix(basis=basis)
hammat

array([[ 0.25,  0.5 ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.5 , -0.75,  0.5 ,  0.5 ,  0.  ,  0.  ],
       [ 0.  ,  0.5 , -0.25,  0.  ,  0.5 ,  0.  ],
       [ 0.  ,  0.5 ,  0.  , -0.25,  0.5 ,  0.  ],
       [ 0.  ,  0.  ,  0.5 ,  0.5 , -0.75,  0.5 ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.5 ,  0.25]])

生成基矢

In [4]:
# 生成具有粒子数生活和动量守恒的基矢
basis = qt.generate.basis.spin_basis(L=4, Nup=2, kblock=0)

# 可以查看基矢的个数：
print("空间维数", basis.Ns)

# # 可以获得某个基矢在全空间中的表示
state = basis.to_full_space(0)

# 可以可视化全空间的基矢：
print("第 0 个基矢：")
basisfull = qt.generate.basis.spin_basis(L=4)
basisfull.show_state(basis[0])

空间维数 2
第 0 个基矢：
↑↑↓↓: (0.5+0j)
↑↓↓↑: (0.5+0j)
↓↑↑↓: (0.5+0j)
↓↓↑↑: (0.5+0j)


In [5]:
# 获得哈密顿量在给定基矢下的矩阵：
mat = ham.to_matrix(basis)
mat

array([[ 0.25      +0.j,  1.06066017+0.j],
       [ 0.70710678+0.j, -0.75      +0.j]])

计算基态能

In [6]:
# 对角化
engs, eigstates = qt.linalg.eigh(mat, k=1)  # 获得最低能量的本征态
engs

array([-1.1160254])

计算纠缠

In [7]:
# 计算纠缠：
entspect = qt.measure.entanglement_spectrum(eigstates[:,0], L//2, basis)  # 纠缠谱
entspect
qt.measure.entropy(entspect)

np.float64(1.1741418371072039)

## 梯子形系统

```
      0   2   4   6
   ---◻---◻---◻---◻---
      |   |   |   |
   ---◻---◻---◻---◻---
      1   3   5   7
```

In [8]:
j1, j2, j3 = 1.0, 2.0, 1.0
import quante as qt
op = qt.generate.operas.spin

L = 5

H_Spart = j1 * op.sum(op.xx(2*i,2*i+2) + op.yy(2*i,2*i+2) + op.zz(2*i,2*i+2) for i in range(L-1))
H_Lpart = op.sum(j2 * (op.xx(2*i+1,2*i+3) + op.yy(2*i+1,2*i+3)) + j1 * op.zz(2*i+1,2*i+3) for i in range(L-1))
H_SLpart = j3 * op.sum(op.xx(2*i,2*i+1) + op.yy(2*i,2*i+1) + op.zz(2*i,2*i+1) for i in range(L))

H = H_Spart + H_Lpart + H_SLpart
H

x 方向和 y 方向都是 zzx 相互作用：

In [9]:
j1, j2, j3 = 1.0, 2.0, 1.0

for L in range(4, 10):
    
    H_Spart = j1 * op.sum(op.xx(2*i,2*i+2) + op.yy(2*i,2*i+2) + op.zz(2*i,2*i+2) for i in range(L-1))
    H_Lpart = op.sum(j2 * (op.xx(2*i+1,2*i+3) + op.yy(2*i+1,2*i+3)) + j1 * op.zz(2*i+1,2*i+3) for i in range(L-1))
    H_SLpart = j3 * op.sum(op.xx(2*i,2*i+1) + op.yy(2*i,2*i+1) + op.zz(2*i,2*i+1) for i in range(L))
    
    H = H_Spart + H_Lpart + H_SLpart
    
    basis = qt.generate.basis.spin_basis(L=2*L, Nup=L)
    mat = H.to_matrix(basis, pauli=False, sparse=True)
    gdeng = qt.linalg.eigvalsh(mat, k=1)[0]
    print(f"L={L}, ground state energy={gdeng}")

L=4, ground state energy=-5.14917530609719
L=5, ground state energy=-6.549539316733603
L=6, ground state energy=-7.967781415010519
L=7, ground state energy=-9.379011867020703
L=8, ground state energy=-10.793329596691645
L=9, ground state energy=-12.206480865439993


与 quspin 的转换

In [10]:
# 对比 quspin 和 quante 的效率（需要在安装 quspin 的环境中运行）
import quante as qt
from quante.bridge.quspin_utils import spin_basis
import time

L = 20
ham = qt.generate.operas.spin.heisenberg_operator(L, j=(1, 1, 1))
ham = ham.expandxy(pauli=False)
quspin_basis = spin_basis(L=L, pauli=0)
lis = ham.to_quspin()

t = time.time()
mat3 = ham.to_matrix(quspin_basis, sparse=True)
print("quspin time: ", time.time()-t)

basis = qt.generate.basis.spin_basis(L=L)
t = time.time()
mat1 = ham.to_matrix(basis, sparse=True)
print("quante time: ", time.time()-t)

print("diff: ",qt.linalg.norm(mat1 - mat3))

quspin time:  1.6671898365020752
quante time:  0.42318224906921387
diff:  0.0


生成矩阵的难点在于, 稀疏矩阵的加法, 它无法利用并行加速

automata 之所以更快是因为, 它最小化了大型稀疏矩阵加法的次数

不同平台本征分解的能力

eigvalsh (real)
|  dim\backend   |  numpy  |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |  matrix ocupied  |  memory needed  |
|:--------------:|:-------:|:-------:|:------------:|:--------:|:------------:|:----------------:|:---------------:|
|  2^14 = 16384  |    ✓    |    ✓    |       ✓      |    ✓     |      ✓       |       2 G        |      4 G        |
|  2^15 = 32768  |    ✓    |    ✓    |       x      |    ✓     |      ✓       |       8 G        |     16 G        |
|  2^16 = 65536  |    x    |    ?    |       x      |    ✓     |      ✓       |      32 G        |     64 G        |

eigh (real)
|  dim\backend   |  numpy  |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |
|:--------------:|:-------:|:-------:|:------------:|:--------:|:------------:|
|  2^14 = 16384  |    ✓    |    ✓    |       ✓      |    ✓     |      ✓       |
|  2^15 = 32768  |    x    |    ?    |       x      |    ✓     |      ✓       |
|  2^16 = 65536  |    x    |    ?    |       x      |    ✓     |      ✓       |

## SU(2) 工具

下面函数中参数中 `jmblock = (J, m)`

`J` 可取的值，可取 `L/2`, `L2/2-1`, ... `0`，`m` 可取的值为 `5`, `4`, `3`, `2`, `1`, `0`, `-1`, `-2`, `-3`, `-4`, `-5`

`dim` 表示子空间的维数，`num` 表示子空间重复的次数，因而：

In [11]:
import quante as qt
L = 4
basis = qt.generate.basis.spin_basis(L, jmblock=(2, 2))

basis.print_dims(L)

   J  |   num  |   dim   
-----------------------
  2.0 |   5    |  1
  1.0 |   3    |  3
  0.0 |   1    |  2
-----------------------
note: \sum num * dim = 2^L


可以与普通的 basis 一样生成矩阵，但目前采用投影矩阵的方法，效率低

In [12]:
ham = qt.generate.operas.spin.heisenberg_operator(L)
mat = ham.to_matrix(basis)
mat.shape, mat

((1, 1), array([[0.75]]))

In [13]:
basis_ = qt.generate.basis.spin_basis(L)
mat_ = ham.to_matrix(basis_)
qt.linalg.eigvalsh(mat_).reshape(4,-1)

array([[-1.6160254 , -0.95710678, -0.95710678, -0.95710678],
       [-0.25      , -0.25      , -0.25      ,  0.1160254 ],
       [ 0.45710678,  0.45710678,  0.45710678,  0.75      ],
       [ 0.75      ,  0.75      ,  0.75      ,  0.75      ]])

可以看到 0.75 确实重复的 5 次

In [14]:
basis = qt.generate.basis.spin_basis(L, jmblock=(1, 1))
mat = ham.to_matrix(basis)
qt.linalg.eigvalsh(mat)

array([-0.95710678, -0.25      ,  0.45710678])

对比可以看到 这三个数每个都重复了三次

验证每个基矢都是 $J^2$ 的本征态

In [15]:
vec = basis.to_full_space(1)  # 第二个基矢，任何一个基矢都满足
# 这个向量是 J^2 的本征态

op = qt.generate.operas
op_Jx = op.sum(op.x(i) for i in range(L))
op_Jy = op.sum(op.y(i) for i in range(L))
op_Jz = op.sum(op.z(i) for i in range(L))
op_J2 = op_Jx**2 + op_Jy**2 + op_Jz**2

basis_ = qt.generate.basis.spin_basis(L)
mat_J2 = op_J2.to_matrix(basis_)

import numpy as np
np.real_if_close(mat_J2 @ vec - 1*(1+1)*vec)  # 这个向量是 J^2 的本征态

array([ 0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])

验证每个基矢都是 $J_z$ 的本征态

In [16]:
basis_ = qt.generate.basis.spin_basis(L)
mat_Jz = op_Jz.to_matrix(basis_)

import numpy as np
np.real_if_close(mat_Jz @ vec - 1*vec)  # 这个向量是 Jz 的本征态

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## 算符代数的一些工具

目前主要是费米子的算符代数，后续会添加玻色子算符代数的内容。

In [17]:
import quante as qt
op = qt.generate.operas.fermion

L = 4
builder = op.builder()
for l in range(L):
    builder += '+-', [l, l+1], 1.
    builder += '+-', [l+1, l], 1.
ham = builder.build()
ham

In [18]:
import quante as qt
op = qt.generate.operas.fermion

builder = op.builder()
builder += '-'*6 + '+'*6, list(range(1, 7))+list(range(1, 7)), 1.
ham = builder.build()
ham


In [19]:
ham.normal_ordering()

## JW transformation

In [20]:
# JW transformation  spin -> fermion
import quante as qt

op = qt.generate.operas.fermion
builder = op.builder()
L = 10
J, γ = 1, 0.0
for i in range(L-1):
    builder += "+-", [i+1, i], (J+γ)/2
    builder += "-+", [i+1, i], -(J-γ)/2
ham = builder.build()
ham.normal_ordering()
# ham.jw_transfer()

In [21]:
# JW transformation  fermion -> spin

import quante as qt

op = qt.generate.operas.fermion
builder = op.builder()
L = 10
J, γ = 1, 0.0
for i in range(L-1):
    builder += "+-", [i+1, i], (J+γ)/2
    builder += "+-", [i, i+1], (J-γ)/2
ham = builder.build()
ham

In [22]:
# 验证正确性：
import quante as qt
import numpy as np

op = qt.generate.operas.spin

L = 5
basis = qt.generate.basis.spin_basis(L=L)

ham = op.heisenberg_operator(L=L).expandxy(pauli=True)

mat1 = ham.to_matrix(basis, pauli=True)

ham = ham.jw_transfer(pauli=True)  # spin -> fermion
ham = ham.jw_transfer()  # fermion -> spin

mat2 = ham.to_matrix(basis)
print(np.linalg.norm(mat1 - mat2))

0.0


In [23]:
# automata - fermion

# 验证正确性：
import quante as qt
import numpy as np

op = qt.generate.operas.spin

L = 5
basis = qt.generate.basis.spin_basis(L=L)
ham = op.heisenberg_operator(L=L).expandxy(pauli=True)
mat1 = ham.to_matrix(basis, pauli=True)
ham = ham.jw_transfer(pauli=True)  # spin -> fermion
ham = ham.jw_transfer()  # fermion -> spin
mpo = ham.to_mpo()
mat2 = mpo.to_matrix().numpy()
print(np.linalg.norm(mat1 - mat2))
print(mpo)

0.0
MPO;  torch.float64;  norm: 1.960e+01;  maxbonddim: 5;  device: cpu;
physdim:    2|    2|    2|    2|    2| 
         ----O-----O-----O-----O-----O----
physdim:    2|    2|    2|    2|    2| 
bonddim:  1     4     5     5     5     1
site:        0     1     2     3     4  


## Spinfull Femion

In [24]:
op = qt.generate.operas.spinful_fermion
builder = op.builder()
builder += '+-|', [0, 1], 1.
builder += '|+-', [1, 0], 1.
builder += '+|-', [1, 0], 1.

In [25]:
import quante as qt
op = qt.generate.operas.spinful_fermion

ham = op.Fermi_Hubbard_operator(L=5)
print(ham)


SpinfulFermionOper (SpinUp | SpinDown) at 0x1b485e42b10, 
|   +     -   |   coef. |   -     +   |   coef. | | +     -       coef. |
|-----------------------|-----------------------|-----------------------|
|   0     1      -1.000 |   0     1       1.000 |   0     1      -1.000 |
|   1     2      -1.000 |   1     2       1.000 |   1     2      -1.000 |
|   2     3      -1.000 |   2     3       1.000 |   2     3      -1.000 |
|   3     4      -1.000 |   3     4       1.000 |   3     4      -1.000 |
| | -     +       coef. |   n   | n       coef. |
|-----------------------|-----------------------|
|   0     1       1.000 |   0     0       5.000 |
|   1     2       1.000 |   1     1       5.000 |
|   2     3       1.000 |   2     2       5.000 |
|   3     4       1.000 |   3     3       5.000 |
|                       |   4     4       5.000 |



In [26]:
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham


In [27]:
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()

from quante.bridge.quspin_utils import spinful_fermion_basis
basis = spinful_fermion_basis(L=L)
mat = ham.to_matrix(basis, dtype=np.float64)
mat

array([[25.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0., 20., -1., ...,  0.,  0.,  0.],
       [ 0., -1., 20., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0., -1.,  0.],
       [ 0.,  0.,  0., ..., -1.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.]])

In [28]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='extend')
ham

In [29]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='extend')

from quante.bridge.quspin_utils import fermion_basis
basis = fermion_basis(L=2*L)
mat = ham.to_matrix(basis, dtype=np.float64)
mat, np.linalg.eigvalsh(mat)[0]

(array([[25.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0., 20., -1., ...,  0.,  0.,  0.],
        [ 0., -1., 20., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0.,  0., ...,  0., -1.,  0.],
        [ 0.,  0.,  0., ..., -1.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]]),
 np.float64(-3.382617960996765))

In [30]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='near')

from quante.bridge.quspin_utils import fermion_basis
basis = fermion_basis(L=2*L)
mat = ham.to_matrix(basis, dtype=np.float64)
mat, np.linalg.eigvalsh(mat)[0]

(array([[25.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0., 20.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0., 20., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]]),
 np.float64(-3.382617960996697))